## Gold
### Modelagem e Disponibilização da Camada de Consumo

*Case Técnico — Engenharia de Dados · iFood*

---

**Objetivo**

Implementar a camada Gold do Lakehouse, agregando os dados da camada Silver em métricas de negócio, para responder às perguntas analíticas do case de forma performática e reutilizável.

**Fonte dos dados**

| | |
|---|---|
| Origem  | ifood_case.silver.trips |
| Período | Janeiro a Maio de 2023 |
| Serviços | Yellow Cab, Green Cab |
| Formato | Delta Table |

**Validações**

- Leitura da camada Silver;
- Agregação de métricas por tipo de táxi, mês e hora do dia;
- Gravação da tabela Gold;
- Consultas de resposta às perguntas de negócio do case.



**1. Instalando e carregando os pacotes**  

In [0]:
from pyspark.sql import functions as F

**2. Leitura da Camada Silver**  


In [0]:
df_silver = (
    spark.table("ifood_case.silver.trips")
    .select(
        "VendorID",
        "tipo",
        "ano_mes",
        "total_amount",
        "passenger_count",
        F.col("pickup_datetime_tratado").alias("pickup_datetime"),
        F.col("dropoff_datetime_tratado").alias("dropoff_datetime")
    )
)

In [0]:
df_silver.count()

In [0]:
df_silver.printSchema()

#### **3. Cálculo das métricas**

- A tabela Gold foi construída no grão (tipo, ano_mes, hora_do_dia), armazenando soma e contagem, e não médias já calculadas, flexibilizando as combinações para agregações.  
- A hora do dia foi extraída de `pickup_datetime`, respeitando o tratamento de timestamps invertidos realizado na camada Silver.

In [0]:
df_gold_metrics = (
    df_silver
    .withColumn("hora_do_dia", F.hour("pickup_datetime"))
    .groupBy("tipo", "ano_mes", "hora_do_dia")
    .agg(
        F.count("*").alias("qtd_corridas"),
        F.sum("total_amount").alias("soma_total_amount"),
        F.sum("passenger_count").alias("soma_passenger_count")
    )
)

display(df_gold_metrics.orderBy("tipo", "ano_mes", "hora_do_dia"))

#### **4. Grava Camada Gold**

##### Trips (Camada de Consumo)

- O enunciado exige que as colunas VendorID, passenger_count, total_amount, pickup_datetime e dropoff_datetime (equivalentes a tpep_pickup_datetime/tpep_dropoff_datetime na fonte Yellow, e
lpep_pickup_datetime/lpep_dropoff_datetime na fonte Green, já padronizadas na Silver) estejam disponíveis na camada de consumo. 
-  Esta tabela mantém o grão de corrida individual, permitindo consultas ad-hoc não previstas neste case.


In [0]:
df_gold_trips = df_silver

In [0]:
(
    df_gold_trips
    .repartition("ano_mes")
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_mes")
    .saveAsTable("ifood_case.gold.trips")
)

In [0]:
%sql
OPTIMIZE ifood_case.gold.trips
ZORDER BY (pickup_datetime)

In [0]:
%sql
COMMENT ON TABLE ifood_case.gold.trips IS
'Camada de consumo com grão de corrida individual, contendo as colunas exigidas pelo case
(VendorID, passenger_count, total_amount, pickup_datetime, dropoff_datetime — equivalentes a
tpep_/lpep_ na origem). pickup_datetime e dropoff_datetime já refletem a correção de timestamps
invertidos realizada na Silver. Complementa ifood_case.gold.trip_metrics, que serve as
perguntas de negócio já agregadas para performance.';

##### Métricas Agregadas
Tabela pré-agregada no grão tipo de táxi, ano/mês e hora do dia, construída para responder de forma performática às perguntas de negócio do case, sem necessidade de reprocessar a camada Silver a cada consulta.

In [0]:
(
    df_gold_metrics
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_mes")
    .saveAsTable("ifood_case.gold.trip_metrics")
)

In [0]:
%sql
COMMENT ON TABLE ifood_case.gold.trip_metrics IS
'Camada Gold com métricas agregadas de corridas por tipo de táxi, mês e hora do dia. Grão:
(tipo, ano_mes, hora_do_dia). Guarda soma e contagem para permitir recombinação correta em 
qualquer agregação superior, evitando o problema de média das médias.
Hora do dia extraída de pickup_datetime_tratado (camada Silver).';
   

#### **5. Perguntas de Negócio**

**Pergunta 1:**     
Qual a média de valor total (total_amount) recebido em um mês considerando todos os yellow táxis da frota?

In [0]:
%sql
SELECT ano_mes
     , ROUND(SUM(soma_total_amount) 
      / SUM(qtd_corridas), 2) AS media_total_amount
 FROM ifood_case.gold.trip_metrics
WHERE tipo = 'yellow'
GROUP BY ano_mes
ORDER BY ano_mes

**Pergunta 2:**       
Qual a média de passageiros (passenger_count) por cada hora do dia que pegaram
táxi no mês de maio, considerando todos os táxis da frota?

In [0]:
%sql
SELECT hora_do_dia
    , ROUND(SUM(soma_passenger_count) 
     / SUM(qtd_corridas), 2) AS media_passenger_count
 FROM ifood_case.gold.trip_metrics
WHERE ano_mes = '2023-05'
GROUP BY hora_do_dia
ORDER BY hora_do_dia